# Ray Unit 4 Capstone - Codespaces Docker Launcher

Use this notebook in GitHub Codespaces for the live Docker-based Ray cluster demo. Codespaces is the Docker host, so there is no Docker Desktop and no separate VM setup in this flow.

This launcher verifies Docker, starts the course Docker Ray cluster, executes the three solution notebooks through Ray Jobs, and shows where the output artifacts are written.

## 1. Show Codespaces Runtime

This cell prints basic Linux runtime information from the Codespaces environment.

In [ ]:
!pwd
!uname -a
!python --version

## 2. Locate The Solution Directory

This cell finds `Ray/4_ray_capstone_project/solution` and changes the notebook working directory to it. The Docker helper script expects to run from this folder.

In [ ]:
from pathlib import Path
import os

start = Path.cwd()
solution_dir = None

for root in [start, *start.parents]:
    candidate = root / "Ray" / "4_ray_capstone_project" / "solution"
    if candidate.exists():
        solution_dir = candidate
        break

if solution_dir is None and Path("/workspaces").exists():
    matches = list(Path("/workspaces").glob("*/Ray/4_ray_capstone_project/solution"))
    if matches:
        solution_dir = matches[0]

if solution_dir is None:
    raise FileNotFoundError("Could not find Ray/4_ray_capstone_project/solution in this Codespace.")

os.chdir(solution_dir)
print("Working directory:", Path.cwd())

## 3. Verify Docker In Codespaces

These commands are the Docker evidence for the live demo. They should show a reachable Docker Engine and Docker Compose command inside Codespaces.

In [ ]:
!docker version
!docker compose version || docker-compose version
!docker info --format 'Engine={{.ServerVersion}}; OS={{.OperatingSystem}}; OSType={{.OSType}}; CPUs={{.NCPU}}'

## 4. Run The Docker-Based Ray Cluster Flow

This starts the course virtual Docker cluster and executes the required notebook flow on Ray through Ray Jobs.

The helper script starts one `ray-head` container and the requested number of `ray-worker` containers. It then runs:

1. `01_download_real_data.ipynb`
2. `02_prepare_assets.ipynb`
3. `03_run_replay.ipynb`

Inside the Ray job, `RAY_ADDRESS=auto`, so the replay notebook connects to the Docker Ray cluster instead of starting isolated local Ray.

In [ ]:
!bash ./run_on_docker_engine.sh --workers 2

## 5. Ray Dashboard

After `ray-head` starts, Codespaces should forward port `8265`. Open the Codespaces **Ports** tab, find port `8265`, and open it in the browser.

The dashboard is useful during the video demo because it shows the Ray head, workers, jobs, and cluster activity.

In [ ]:
!cd ../../1_cluster_setup && (docker compose ps || docker-compose ps)
!docker ps --filter "name=ray" --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'

## 6. Inspect Output Artifacts

The Docker run writes persistent files under the head container's mounted workspace. This folder remains visible in Codespaces after the Ray job finishes.

In [ ]:
!find ../../1_cluster_setup/head_workspace/ray_capstone -maxdepth 3 -type f | sort | head -80

## 7. Stop The Cluster After The Demo

Run this cleanup cell only after you finish inspecting the dashboard and artifacts.

In [ ]:
# Uncomment after the demo if you want to stop the containers.
# !cd ../../1_cluster_setup && (docker compose down || docker-compose down)